In [7]:
from sklearn.datasets import load_diabetes

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [8]:
X,y = load_diabetes(return_X_y=True)

In [9]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=2)

In [11]:
lnr = LinearRegression()
lnr.fit(X_train, y_train)

LinearRegression()

In [12]:
print(lnr.coef_)
print(lnr.intercept_)

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


In [13]:
y_pred = lnr.predict(X_test)

In [14]:
r2 = r2_score(y_test,y_pred)
r2

0.4399338661568969

### Now, doing the same from scratch

In [15]:
class MiniBatchGD:

    def __init__(self, learning_rate, epochs, batch_size):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X_train, y_train):

        n_samples, n_features = X_train.shape

        # Initialize parameters
        # β = [β₁, β₂, ..., βₙ]
        self.coef_ = np.ones(n_features)

        # β₀ (intercept)
        self.intercept_ = 0

        for epoch in range(self.epochs):

            # Shuffle data every epoch
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]

            # Process data in mini-batches
            for start in range(0, n_samples, self.batch_size):

                end = start + self.batch_size

                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                # Forward pass
                # ŷ = X_batch β + β₀
                y_hat = np.dot(X_batch, self.coef_) + self.intercept_

                # Errors
                # e = y_batch - ŷ
                errors = y_batch - y_hat

                m = X_batch.shape[0]

                # Gradients
                # ∂L/∂β₀ = -(2/m) Σ(y - ŷ)
                intercept_der = -2 * np.mean(errors)

                # ∂L/∂β = -(2/m) Xᵀ(y - ŷ)
                coef_der = -2 * np.dot(errors, X_batch) / m

                # Parameter updates
                # β₀ := β₀ - α(∂L/∂β₀)
                self.intercept_ -= self.lr * intercept_der

                # β := β - α(∂L/∂β)
                self.coef_ -= self.lr * coef_der

    def predict(self, X_test):
        # ŷ = Xβ + β₀
        return np.dot(X_test, self.coef_) + self.intercept_

In [16]:
model = MiniBatchGD(0.01,40,40)

In [17]:
model.fit(X_train, y_train)

In [18]:
model.predict(X_test)

print(f"Slope (m): {model.coef_}")
print(f"Intercept (b): {model.intercept_}")

Slope (m): [ 6.78402748  2.0426943  16.01511655 12.81784583  6.32005972  5.01766691
 -8.50492513 10.97010961 16.09223069 10.13693288]
Intercept (b): 150.4529415156777


In [19]:
# Now, check the R2 Score
y_pred = model.predict(X_test)

In [20]:
r2 = r2_score(y_test,y_pred)
r2

0.03479028316509958